# Experiment 5.1 — Boundary-free temporal decoder

Analysis-only notebook. Training and probes are produced by the Slurm workflow. Configuration selection uses validation balanced accuracy only; test balanced accuracy is reported only after selection.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_5_1_boundary_free_temporal_decoder' / 'frozen_local_leaky_interface_rsnn_v1'
runs = pd.read_csv(ART / 'runs.csv')
interface_probes = pd.read_csv(ART / 'interface_probes.csv')
histories = pd.read_csv(ART / 'histories.csv')
local_reference = pd.read_csv(ART / 'local_reference.csv')
manifest = json.loads((ART / 'manifest.json').read_text(encoding='utf-8'))
display(manifest)
display(runs.head())

## 1. Aggregate native decoder performance

Mean ± SD is computed across the three frozen-local seeds.

In [ ]:
native_summary = (
    runs.groupby(['interface', 'architecture', 'tau_mem_ms'])
    .agg(
        val_ba_mean=('native_val_balanced_accuracy', 'mean'),
        val_ba_sd=('native_val_balanced_accuracy', 'std'),
        test_ba_mean=('native_test_balanced_accuracy', 'mean'),
        test_ba_sd=('native_test_balanced_accuracy', 'std'),
        hidden_rate_mean=('native_test_hidden_events_per_neuron_second', 'mean'),
        hidden_tail_mean=('native_test_hidden_tail_event_fraction', 'mean'),
    )
    .reset_index()
)
display(native_summary.sort_values(['architecture', 'interface', 'tau_mem_ms']))

## 2. Validation-only tau selection

For each interface × architecture, select tau only by mean validation native BA. Test metrics are joined only after the selection row is fixed.

In [ ]:
selection = (
    native_summary.sort_values(['interface', 'architecture', 'val_ba_mean', 'tau_mem_ms'], ascending=[True, True, False, True])
    .groupby(['interface', 'architecture'], as_index=False)
    .first()
)
selection['test_ba_pct'] = 100 * selection['test_ba_mean']
display(selection[['interface', 'architecture', 'tau_mem_ms', 'val_ba_mean', 'test_ba_mean', 'test_ba_sd']])

## 3. Native BA versus temporal memory

This plot answers whether leaky242 changes the temporal decoder ceiling and whether recurrence adds value beyond passive membrane state.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for (interface, architecture), group in native_summary.groupby(['interface', 'architecture']):
    group = group.sort_values('tau_mem_ms')
    ax.errorbar(group['tau_mem_ms'], 100 * group['test_ba_mean'], yerr=100 * group['test_ba_sd'], marker='o', capsize=3, label=f'{interface} / {architecture}')
ax.set_xlabel('Temporal tau_mem (ms)')
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_title('Exp5.1 native analog-head sequence decoding')
ax.legend()
fig.tight_layout()
plt.show()

## 4. Paired effect of boundary-free pooling

Pair raw-L2 and leaky242 by seed within every architecture/tau. Positive values mean leaky242 improves native BA.

In [ ]:
pivot = runs.pivot_table(index=['architecture', 'tau_mem_ms', 'seed'], columns='interface', values='native_test_balanced_accuracy').reset_index()
pivot['leaky_minus_raw'] = pivot['leaky242'] - pivot['raw_l2']
paired_pooling = (
    pivot.groupby(['architecture', 'tau_mem_ms'])['leaky_minus_raw']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
display(paired_pooling)
fig, ax = plt.subplots(figsize=(9, 5))
for architecture, group in paired_pooling.groupby('architecture'):
    group = group.sort_values('tau_mem_ms')
    ax.errorbar(group['tau_mem_ms'], 100 * group['mean'], yerr=100 * group['std'], marker='o', capsize=3, label=architecture)
ax.axhline(0, linewidth=1)
ax.set_xlabel('Temporal tau_mem (ms)')
ax.set_ylabel('Leaky242 - raw L2 test BA (points)')
ax.set_title('Paired effect of boundary-free pooling')
ax.legend()
fig.tight_layout()
plt.show()

## 5. Does leaky242 preserve phase-accessible information?

Interface probes are independent of FF/RSNN training. The key comparison is raw-L2 versus leaky242 for Fixed250/Relative10 accessibility and endpoint state.

In [ ]:
interface_summary = (
    interface_probes.groupby(['interface', 'probe_type'])['test_ba']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
display(interface_summary)
plot = interface_summary.pivot(index='probe_type', columns='interface', values='mean')
ax = (100 * plot).plot(kind='bar', figsize=(11, 6), rot=25)
ax.set_ylabel('Test balanced accuracy (%)')
ax.set_title('Linear accessibility at the L2-to-temporal-decoder interface')
fig = ax.get_figure()
fig.tight_layout()
plt.show()

## 6. Representation flow at validation-selected configurations

Compare the historical frozen L2 reference with interface and temporal-hidden probes. This is the main information-flow diagnostic.

In [ ]:
local_metrics = pd.DataFrame({
    'stage': ['L2 FullCount', 'L2 Fixed250', 'L2 Relative10'],
    'test_ba': [
        local_reference['full_count_test_ba'].mean(),
        local_reference['fixed250_ordered_test_ba'].mean(),
        local_reference['relative10_ordered_test_ba'].mean(),
    ],
})
selected_rows = []
for _, selected in selection.iterrows():
    subset = runs[(runs.interface == selected.interface) & (runs.architecture == selected.architecture) & (runs.tau_mem_ms == selected.tau_mem_ms)]
    for metric, label in [
        ('hidden_whole_count_test_ba', 'Hidden WholeCount'),
        ('hidden_fixed250_ordered_test_ba', 'Hidden Fixed250'),
        ('hidden_relative10_ordered_test_ba', 'Hidden Relative10'),
        ('hidden_uend_test_ba', 'Hidden Uend'),
    ]:
        selected_rows.append({'interface': selected.interface, 'architecture': selected.architecture, 'tau_mem_ms': selected.tau_mem_ms, 'stage': label, 'test_ba': subset[metric].mean()})
selected_probe_table = pd.DataFrame(selected_rows)
display(local_metrics)
display(selected_probe_table)

## 7. State consolidation diagnostic

For RSNN, compare hidden Uend with hidden WholeCount and Relative10 across tau. If Uend approaches Relative10, recurrent state has consolidated temporal information that otherwise requires trajectory access.

In [ ]:
rsnn = runs[runs.architecture == 'rsnn']
probe_summary = (
    rsnn.groupby(['interface', 'tau_mem_ms'])[[
        'hidden_whole_count_test_ba',
        'hidden_relative10_ordered_test_ba',
        'hidden_uend_test_ba',
    ]]
    .mean()
    .reset_index()
)
display(probe_summary)
for interface, group in probe_summary.groupby('interface'):
    fig, ax = plt.subplots(figsize=(9, 5))
    group = group.sort_values('tau_mem_ms')
    ax.plot(group['tau_mem_ms'], 100 * group['hidden_whole_count_test_ba'], marker='o', label='Hidden WholeCount')
    ax.plot(group['tau_mem_ms'], 100 * group['hidden_relative10_ordered_test_ba'], marker='o', label='Hidden Relative10')
    ax.plot(group['tau_mem_ms'], 100 * group['hidden_uend_test_ba'], marker='o', label='Hidden Uend')
    ax.set_xlabel('Temporal tau_mem (ms)')
    ax.set_ylabel('Test balanced accuracy (%)')
    ax.set_title(f'RSNN state consolidation — {interface}')
    ax.legend()
    fig.tight_layout()
    plt.show()

## 8. Validation learning curves for the globally selected configuration

Global selection is based only on mean validation native BA. Curves show mean ± SD over seeds.

In [ ]:
global_selected = native_summary.sort_values(['val_ba_mean', 'tau_mem_ms'], ascending=[False, True]).iloc[0]
curve = histories[(histories.interface == global_selected.interface) & (histories.architecture == global_selected.architecture) & (histories.tau_mem_ms == global_selected.tau_mem_ms)]
curve_summary = curve.groupby('epoch')['val_balanced_accuracy'].agg(['mean', 'std']).reset_index()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(curve_summary['epoch'], curve_summary['mean'], label='Mean val BA')
ax.fill_between(curve_summary['epoch'], curve_summary['mean'] - curve_summary['std'], curve_summary['mean'] + curve_summary['std'], alpha=0.2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation balanced accuracy')
ax.set_title(f"Selected by validation: {global_selected.interface} / {global_selected.architecture} / tau={global_selected.tau_mem_ms:.0f} ms")
ax.legend()
fig.tight_layout()
plt.show()

## Interpretation gates

1. `leaky242 > raw_l2` in paired native BA: boundary-free aggregation reduces temporal-decoder burden.
2. Leaky242 interface Fixed250/Relative10 probe near raw-L2: smoothing preserves the useful local/phase representation.
3. `RSNN > FF`: learned recurrence adds value beyond passive membrane memory.
4. Hidden Uend approaching hidden Relative10: recurrent state consolidates phase/history into the current state.
5. Strong hidden probes but weak native analog head: decoder representation is good but class-evidence readout remains limiting.